# 🚁 YOLOv8 Testing on Images and Videos
### Run the cells in order

In [ ]:
# Cell 1: Installation
!pip install ultralytics -q
print('✅ Installation complete!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.3 MB/s eta 0:00:00
✅ Installation complete!


In [ ]:
# Cell 2: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('\n📂 Your files in Drive:')
!ls /content/drive/MyDrive/

Mounted at /content/drive

📂 Your files in Drive:
 best_sky.pt	    result_video.mp4   test2.jpeg   v12.mp4   v3.mp4
 best_sky_v2.pt     t1.jpg	       test3.jpeg   v15.mp4   v5.mp4
 best_unified.pt    t2.jpg	       test4.jpg    v16.mp4   v7.mp4
'Colab Notebooks'   t3.jpeg	       test6.jpeg   v17.mp4   v8.mp4
 i1.jpg		    t4.jpeg	       v10.mp4	    v1.mp4    v9.mp4
 result_image.jpg   test1.jpeg	       v11.mp4	    v2.mp4


In [ ]:
# Cell 3: Load Model
from ultralytics import YOLO

# You can use best_sky.pt or yolov8x.pt
# If you have best_sky.pt in Drive:
# model = YOLO('/content/drive/MyDrive/best_sky.pt')

# Or use the pre-trained model:
model = YOLO('yolov8x.pt')
print('✅ Model loaded!')
print(f'📋 Classes: {model.names}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Model loaded!
📋 Classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 4

In [ ]:

# Cell 4: Detection on Image
import cv2
import matplotlib.pyplot as plt
import os

# ✏️ Change the image name
IMAGE_NAME = 't3.jpeg'
IMAGE_PATH = f'/content/drive/MyDrive/{IMAGE_NAME}'
CONF       = 0.25
ALLOWED    = ['airplane', 'helicopter', 'bird', 'kite']

if not os.path.exists(IMAGE_PATH):
    print(f'❌ Image not found: {IMAGE_NAME}')
else:
    results = model(IMAGE_PATH, conf=CONF, verbose=False)
    frame   = cv2.imread(IMAGE_PATH)
    count   = 0
    h, w    = frame.shape[:2]

    for box in results[0].boxes:
        cls_id     = int(box.cls[0])
        class_name = model.names[cls_id]
        conf_val   = float(box.conf[0])

        # Filter classes
        if class_name not in ALLOWED:
            continue

        x1,y1,x2,y2 = map(int, box.xyxy[0])

        # Ignore large bounding boxes
        if (x2-x1)*(y2-y1) > h*w*0.8:
            continue

        count += 1
        cx,cy  = (x1+x2)//2, (y1+y2)//2

        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 3)
        label = f'{class_name}: {conf_val*100:.0f}%'
        cv2.putText(frame, label, (x1, max(y1-10,20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
        cv2.circle(frame, (cx,cy), 5, (0,255,0), -1)
        cv2.putText(frame, f'({cx},{cy})', (x1, y2+25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    # Save and display
    cv2.imwrite('/content/drive/MyDrive/result_image.jpg', frame)

    plt.figure(figsize=(14,8))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(f'🚁 Detected {count} object(s)', fontsize=16)
    plt.axis('off')
    plt.show()
    print(f'✅ Object count: {count}')
    print(f'💾 Saved: result_image.jpg')

In [ ]:
# Cell 5: Detection on Video
import shutil
from IPython.display import HTML, display
from base64 import b64encode

# ✏️ Change the video name
VIDEO_NAME = 'v15.mp4'
VIDEO_PATH = f'/content/drive/MyDrive/{VIDEO_NAME}'
CONF       = 0.25
MAX_FRAMES = 9999

if not os.path.exists(VIDEO_PATH):
    print(f'❌ Video not found: {VIDEO_NAME}')
else:
    shutil.copy(VIDEO_PATH, '/content/input.mp4')

    cap    = cv2.VideoCapture('/content/input.mp4')
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = max(int(cap.get(cv2.CAP_PROP_FPS)), 25)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'📹 {width}x{height} @ {fps}fps | {total} frames')

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter('/content/output_raw.mp4', fourcc, fps, (width, height))

    frame_count  = 0
    drone_frames = 0

    while cap.isOpened() and frame_count < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=CONF, verbose=False)
        count   = 0

        for box in results[0].boxes:
            cls_id     = int(box.cls[0])
            class_name = model.names[cls_id]
            conf_val   = float(box.conf[0])
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            count += 1
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 3)
            label = f'{class_name}: {conf_val*100:.0f}%'
            cv2.putText(frame, label, (x1, max(y1-10,20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

        if count > 0:
            drone_frames += 1
            cv2.putText(frame, f'DETECTED! ({count})', (20,50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 3)

        cv2.putText(frame, f'Frame:{frame_count}/{min(MAX_FRAMES,total)}',
                    (20, height-20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

        out.write(frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f'  ⏳ {frame_count}/{min(MAX_FRAMES,total)} frames...')

    cap.release()
    out.release()

    !ffmpeg -i /content/output_raw.mp4 -vcodec libx264 /content/final.mp4 -y -loglevel quiet
    shutil.copy('/content/final.mp4', '/content/drive/MyDrive/result_video.mp4')

    print(f'\n✅ Done! {drone_frames}/{frame_count} frames')
    print(f'💾 Saved: result_video.mp4')

    mp4      = open('/content/final.mp4','rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))

📹 1280x720 @ 25fps | 885 frames
  ⏳ 30/885 frames...
  ⏳ 60/885 frames...
  ⏳ 90/885 frames...
  ⏳ 120/885 frames...
  ⏳ 150/885 frames...
  ⏳ 180/885 frames...
  ⏳ 210/885 frames...
  ⏳ 240/885 frames...
  ⏳ 270/885 frames...
  ⏳ 300/885 frames...
  ⏳ 330/885 frames...
  ⏳ 360/885 frames...
  ⏳ 390/885 frames...
  ⏳ 420/885 frames...
  ⏳ 450/885 frames...
  ⏳ 480/885 frames...
  ⏳ 510/885 frames...
  ⏳ 540/885 frames...
  ⏳ 570/885 frames...
  ⏳ 600/885 frames...
  ⏳ 630/885 frames...
  ⏳ 660/885 frames...
  ⏳ 690/885 frames...
  ⏳ 720/885 frames...
  ⏳ 750/885 frames...
  ⏳ 780/885 frames...
  ⏳ 810/885 frames...
  ⏳ 840/885 frames...
  ⏳ 870/885 frames...
